In [23]:
import gpytorch
import torch
from laplace_model_selection.metrics import MLL
from helpers import gp_classes, example_kernels
import numpy as np

In [24]:
torch.set_default_dtype(torch.float64)

In [25]:
example_kernels.available()

['LIN',
 'MAT32',
 'MAT52',
 'PER',
 'RQ',
 'SE',
 'C*LIN',
 'C*PER',
 'C*SE',
 'C*C*SE',
 'LIN*PER',
 'LIN*SE',
 'MAT32+MAT52',
 'MAT32+PER',
 'MAT32*PER',
 'MAT32+SE',
 'MAT32*SE',
 'MAT52*PER',
 'MAT52+SE',
 'PER*SE',
 'PER+C*SE',
 'SE+SE',
 'SE*SE',
 'SE+SE+SE',
 'MAT32+(MAT52*PER)',
 'PER*(SE+RQ)',
 '[LIN; LIN]',
 '[LIN;* LIN]',
 '[LIN; SE]',
 '[SE;* 1]',
 '[PER;* 1]',
 '[SE; SE]',
 '[SE;* SE]',
 '[SE+SE; 1]',
 '[SE+SE; 1]_noadd',
 '[SE; LIN]']

In [26]:
train_x = torch.linspace(0, 1, 100)
train_y = torch.sin(train_x * (2 * torch.pi)) + 0.1 * torch.randn(train_x.size())
likelihood = gpytorch.likelihoods.GaussianLikelihood()
model = gp_classes.ExactGPModel(train_x, train_y, likelihood, kernel_name="MAT52")

In [27]:
mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)
mll(model(train_x), train_y).item()*100

-87.29399334871616

In [28]:
pred = likelihood(model(train_x))
torch.distributions.MultivariateNormal(pred.mean, pred.covariance_matrix).log_prob(train_y).item()

-87.29399334871617

In [32]:
A = - 0.5*train_y.T @ pred.covariance_matrix.inverse() @ train_y 
B = - 0.5*torch.logdet(pred.covariance_matrix)
C = - 0.5*train_y.size(0)*np.log(2*np.pi)

A + B + C

tensor(-87.2940, grad_fn=<AddBackward0>)